In [4]:
import os
import pandas as pd
import re

In [8]:
df = pd.DataFrame(columns = ["Test positive", "Test negative", "Dataset", "Confidence weight", "Support weight"])
for file in os.listdir("hyperparam_tuning_results/"):
    if file.endswith(".txt"):
        with open(os.path.join("hyperparam_tuning_results/", file), "r") as f:
            lines = f.readlines()
            if len(lines) != 7:
                print(f"Skipping file {file} due to unexpected number of lines: {len(lines)}")
                print(lines)
                continue 
            test_pos_line = lines[4]
            test_neg_line = lines[5]
            test_pos = float(test_pos_line.split(": ")[1].strip())
            test_neg = float(test_neg_line.split(": ")[1].strip())
            dataset = file.split("_SDIGA")[0]
            # File name example: synthetic_SDIGA_mg_cp_mp_cw0.5_sw0.3
            cw_match = re.search(r'cw(0\.\d+)', file)
            sw_match = re.search(r'sw(0\.\d+)', file)
            if cw_match and sw_match:
                cw = float(cw_match.group(1))
                sw = float(sw_match.group(1))
            else:
                print(f"Skipping file {file} due to missing cw or sw in filename")
                continue
            df.loc[len(df)] = [test_pos, test_neg, dataset, cw, sw]
df["Youden"] = df["Test positive"] - df["Test negative"]

In [9]:
df.head()

,Test positive,Test negative,Dataset,Confidence weight,Support weight,Youden
0,0.280000,0.035000,synthetic,0.5,0.4,0.245000
1,1.000000,0.173913,brca_n,0.4,0.5,0.826087
2,0.373750,0.231667,synthetic,0.4,0.4,0.142083
3,0.001389,0.000000,polluted_synthetic,0.5,0.5,0.001389
4,0.121366,0.002245,Facebook,0.5,0.5,0.119121


In [11]:
df.groupby("Dataset").apply(lambda x: x.loc[x["Youden"].idxmax()])


/tmp/ipykernel_1104835/2288029840.py:1: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df.groupby("Dataset").apply(lambda x: x.loc[x["Youden"].idxmax()])


,Test positive,Test negative,Dataset,Confidence weight,Support weight,Youden
Dataset,,,,,,
Facebook,0.268169,0.007056,Facebook,0.3,0.5,0.261113
brca_n,1.000000,0.086957,brca_n,0.5,0.4,0.913043
polluted_synthetic,0.220833,0.032353,polluted_synthetic,0.3,0.4,0.188480
synthetic,0.280000,0.035000,synthetic,0.5,0.4,0.245000
